# QC-Py-Cloud-01 — Risk Parity Composite Multi-Asset

**Module**: Hands-On AI Trading, Ch.10 — Risk Parity & Portfolio Construction

**Objectif**: Implementer une allocation Risk Parity (ponderation inverse-volatilite) sur un univers multi-actifs, avec filtres de tendance pour eviter les actifs en phase baissiere.

**Approche cloud-native**: L'algorithme est execute sur QuantConnect Cloud. Les resultats sont syncs ci-dessous.

## 1. Concept : Risk Parity

Le Risk Parity ("parite des risques") est une approche d'allocation developpee par Bridgewater (Ray Dalio, 1996). Plutot que d'allouer un capital egal par position (equal-weight), on alloue un **budget de risque egal**.

**Principe**:
- Chaque actif recoit un poids inversement proportionnel a sa volatilite
- Les actifs peu volatils (obligations) recoivent plus de capital que les actifs volatils (actions)
- Objectif : chaque actif contribue egalement au risque du portefeuille

**Avantages**:
- Diversification naturelle跨 actifs
- Meilleur ratio rendement/risque en theorie
- Resilience en cas de crise equity

**Limites**:
- Sous-performance quand les taux montent (TLT bear 2020-2024)
- necessite un univers diversifie pour fonctionner
- Leverage implicite sur les actifs low-vol

## 2. Univers : 6 ETFs Multi-Asset

| Ticker | Classe d'actif | Role dans le portefeuille |
|--------|---------------|--------------------------|
| SPY | Actions US (S&P 500) | Growth, equity risk premium |
| TLT | Obligations US (20y+) | Duration, safe haven |
| GLD | Or | Inflation hedge, crisis alpha |
| EFA | Actions internationales | Diversification geographique |
| EEM | Marches emergents | Growth, carry |
| DBC | Matieres premieres | Inflation, commodity cycle |

## 3. Methodologie

Quatre versions ont ete testees sur QC Cloud (projet 30820857):

### v1 — Risk Parity pur
- Ponderation inverse-volatilite stricte
- Drawdown cap 12% + trailing stop 4%
- Rebalance mensuel

### v2 — Risk Parity + filtre momentum
- Seuls les actifs avec momentum 12m positif sont eligibles
- Inverse-volatilite parmi les candidats
- Pas de drawdown cap / trailing stop

### v3 — Rotation tactique + vol targeting
- Allocation egale parmi les actifs momentum-positif
- Scaling volatilite pour cibler 8% de vol annuelle
- Rebalance toutes les 3 semaines

### v4 — SMA200 + 6m Momentum (AQR Hybrid)
- Double filtre : prix > SMA200 ET momentum 6m > 0
- Allocation egale parmi les candidats qualifies
- Rebalance mensuel
- Approche inspiree de Hurst, Ooi, Pedersen (AQR, 2014)

## 4. Resultats QC Cloud

**Projet QC Cloud**: 30820857 (`Cloud-RiskParity-Composite`)
**Periode**: 2014-01-01 au 2025-01-01 (11 ans)
**Capital initial**: $100,000

| Version | Sharpe | CAGR | MaxDD | Net Profit | Ordres | Win Rate |
|---------|--------|------|-------|------------|--------|----------|
| v1 (Risk Parity pur) | -0.913 | -0.41% | 12.2% | -4.4% | 146 | 73% |
| v2 (+ filtre momentum) | 0.178 | 4.77% | 26.0% | +67.0% | 531 | 76% |
| v3 (+ vol targeting) | -0.052 | 2.52% | 15.7% | +31.6% | 741 | 76% |
| **v4 (SMA200+Mom6m)** | **0.278** | **6.17%** | **20.4%** | **+93.3%** | **474** | **76%** |

### Benchmark : SPY Buy & Hold
Sur la meme periode, SPY a un CAGR ~12.8% et un MaxDD ~24%.

## 5. Analyse : Pourquoi le Risk Parity ne atteint pas Sharpe 0.8

La cible de Sharpe > 0.8 n'est pas atteinte (meilleur resultat : 0.278). Voici pourquoi :

### Cause 1 : TLT en bear market seculaire (2020-2024)
Les obligations long terme (TLT) ont perdu ~50% de leur valeur pendant le cycle de hausse des taux Fed (2020-2024). Le risk parity surpondere naturellement TLT (faible volatilite historique), ce qui a detruit la performance.

### Cause 2 : Matieres premieres et emergents faibles
DBC (commodites) et EEM (marches emergents) ont eu une "decennie perdue" (2014-2024). Ces actifs contribuent peu de rendement mais beaucoup de volatilite, diluant le Sharpe.

### Cause 3 : Diversification excessive
Avec 6 actifs dont seulement 2 performants (SPY, GLD), la diversification dilue l'alpha. Les strategies qui atteignent Sharpe > 0.8 (comme l'EMA-Cross-Alpha a 0.996) sont concentrees sur un univers equity.

### Cause 4 : Drawdown cap et trailing stop contre-productifs
La v1 (avec stops) a un Sharpe de -0.913. Les mecanismes de protection empechent la recuperation apres les drawdowns, transformant les pertes temporaires en pertes permanentes.

### Lecon : Le Risk Parity est un outil d'allocation, pas d'alpha
Le risk parity excelle pour construire un portefeuille diversifie avec un bon ratio rendement/risque *ajuste pour la diversification*. Mais il ne genere pas d'alpha pur. Pour ameliorer le Sharpe, il faut combiner avec des signaux d'alpha (momentum, carry, value).

## 6. Code source (v4 — meilleur resultat)

Le code est disponible sur QC Cloud (projet 30820857). Le notebook local ne contient que les resultats et l'analyse, conformement au workflow cloud-native.

```python
# Lien QC Cloud : https://www.quantconnect.com/project/30820857
# Approche : SMA200 + 6m momentum dual filter
# Rebalance mensuel, allocation egale parmi actifs eligibles
```

## 7. Pour aller plus loin

1. **Risk Parity + Alpha Overlay**: Ajouter des signaux d'alpha (momentum score) pour ponderer au lieu de l'egalite
2. **Dynamic Vol Targeting**: Ajuster l'exposition globale selon le regime de vol (VIX > 25 = reduce)
3. **Carry Signal**: Integrer le carry (dividend yield, bond yield) comme signal complementaire
4. **Universe etendu**: Ajouter des ETFs sectoriels (XLU, XLK), crypto (BTC), ou real estate (VNQ)

**Reference**: Hurst, Ooi, Pedersen (2014) — "A Century of Evidence on Trend-Following Investing", AQR.

In [ ]:
# Cellule Cloud-only : le code est execute sur QC Cloud, pas localement
# Voir les resultats dans la section 4 ci-dessus
print("Notebook QC-Py-Cloud-01 — Resultats sync depuis QC Cloud projet 30820857")